# Step 5: Statistical tests from the proposal

**Handover notes.** This notebook is empty on purpose. The markdown cells below describe what each section should
do and the decisions already made. Add code cells under each heading.

Run it after the pipeline scripts (`01` to `04`, see `CLAUDE.md`). Read **only** `data/processed/model_table.parquet`
(one row per zone × hour, 263 zones × 8,759 hours, weather already joined). Don't touch the raw CSVs.

Environment:
- `pyarrow` is **not installed**, so pandas can't read Parquet. Load it with DuckDB
  (`duckdb.connect().execute("SELECT ... FROM 'path/to/model_table.parquet'").df()`).
- DuckDB 2.0: use `con.execute(sql).fetchall()` or `.df()`. Relations don't have `fetchone()`.
- Paths are relative to `scripts/`, so the data is at `../data/processed/`.
- Strip outputs before committing (`nbstripout --install` once).

Sections, one per proposal technique:

| Proposal technique | Section |
|---|---|
| Linear regression | Model ladder |
| Heteroskedasticity and residual plots | Residual plots, QQ plots, Breusch-Pagan |
| F-test | F-tests on the model ladder |
| Collinearity | F-tests (joint vs individual, auxiliary regressions). **No VIF**: the professor doesn't allow it |
| Comparing models | Model comparison table |
| Forecast accuracy | MAE / RMSE on the test set |
| Frisch-Waugh-Lovell | FWL weather effect |

## Settings and data

Decisions already made (change them if you have a reason, but write the reason down):

- **Calendar columns:** derive hour of day, day of week (ISO, 1 = Monday), month, a weekend flag and the date from
  the `hour` timestamp. DuckDB has `hour()`, `isodow()`, `month()`, and `hour::DATE` gives the date.
- **Zones:** model only the busiest ~50 zones, ranked by pickups in the **training** data. The top 60 carry about
  90% of all pickups, and most other zones are mostly zeros. FWL uses the city-wide total of all 263 zones.
- **Target:** `log(pickups + 1)`. Count variance grows with the mean (see the heteroskedasticity section).
- **Split:** a strict date split leaves the test months with no month coefficient. Instead, within **every month**:
  last 7 days = **test**, the 7 days before = **validation**, the rest = **train**. Christmas and Thanksgiving fall
  in test weeks, so expect larger errors there.
- **Baseline:** for each zone-hour, the pickups in the same zone at the same clock hour 7 days earlier (a self-join).
- **Missing weather:** only 2025-12-31 23:00 has blank weather (from the end-of-hour shift). Drop that row.
- **Weather columns:** `temperature_2m, precipitation, snowfall, snow_depth, cloud_cover, shortwave_radiation, is_day`.
  Check the exact names in the parquet. They change if step 1 is rerun with a different `--precip`.
- Weather is the same for every zone in an hour, so weather effects come from ~8,759 hours, not from every row.
  Use standard errors **clustered by day** for the zone panel and **HAC** (e.g. 24 lags) for the hourly city series.

## Linear regression: the model ladder

Fit a ladder of OLS models on the **training** rows. Each model adds one group of predictors to the one before,
so each pair is **nested** (the F-tests need that):

| Model | Adds |
|---|---|
| M1 | hour of day (dummies) |
| M2 | day of week (dummies) |
| M3 | month (dummies) |
| M4 | pickup zone (dummies) |
| M5 | weather (the 7 columns above) |
| M6 | a separate daily curve for weekends (hour × weekend dummies; leave out hour 0 so they don't duplicate the day-of-week dummies) |

Report the number of parameters and R² for each model. Also report the M5 weather coefficients with
**day-clustered** standard errors. In a log model, coefficient × 100 ≈ % change in pickups.

Hint: `statsmodels.formula.api.ols` with `C(...)` for categories.

## Heteroskedasticity: residual plots and QQ plots

Do this **before** the F-tests, because it decides which standard errors they can trust.

Fit the fullest model twice, on raw `pickups` and on `log(pickups + 1)`, and show for each:

- **Residuals vs fitted** (use a random sample of ~20k points so the plot stays readable). A funnel shape means
  heteroskedasticity.
- **Residual SD by fitted-value decile.** A rising line means the errors grow with demand.
- **QQ plot** of the residuals (`sm.qqplot`). An S-shape or bent tails mean heavy tails or skew.
- **Breusch-Pagan test** (`statsmodels.stats.diagnostic.het_breuschpagan`). H₀ = constant error variance.

With ~230k rows the test rejects even for small departures, so judge the size of the problem from the plots.

What to expect: on raw counts the spread grows with demand, and the model can predict negative pickups. The log
reverses this: low-demand hours become the noisiest. Neither has constant variance, so back up the F-tests with
robust (clustered / HAC) standard errors. A square-root target is a middle option worth comparing.

## F-test: does each added group of predictors improve the model?

Test each model in the ladder against the one before it (`statsmodels.stats.anova.anova_lm(*models)`).
H₀ = the added coefficients are all zero. A large F (small p) means the group explains significantly more.

The classic F-test assumes equal variance, which fails here. So also test the weather block with a **robust Wald
test** on the day-clustered M5 (`model.wald_test("a = 0, b = 0, ...")`).

## Collinearity: F-tests (no VIF)

Collinearity means two or more predictors carry almost the same information. Their coefficients get large standard
errors and can flip sign. **Don't use VIF.** Use these two F-test checks on the weather variables:

1. **Joint vs individual tests.** For suspect pairs, e.g. (`is_day`, `shortwave_radiation`),
   (`cloud_cover`, `shortwave_radiation`) and (`precipitation`, `snowfall`), compare each coefficient's own p-value
   with a joint F-test that both are zero (day-clustered M5). If the joint test rejects but the individual tests
   don't, the pair is collinear.
2. **Auxiliary regressions (Farrar-Glauber F-test).** Use one row per training hour (weather doesn't vary by zone).
   Regress each weather variable on the calendar dummies plus the other weather variables. Report the auxiliary R²,
   the overall F, and a **partial F** (calendar only vs calendar + other weather), which asks whether the other
   weather still predicts it once the calendar is accounted for. With ~4,700 hours the F-tests reject easily, so
   judge the size from R²: above ~0.8 is serious.

Also print the correlation matrix of the weather variables for reference.

What to expect: `temperature_2m`, `is_day` and `shortwave_radiation` are mostly explained by hour × month. If
`precipitation` and `snowfall` are highly collinear, rerun step 1 with `--precip rain`.

## Comparing models

Make one table with a row per model in the ladder:
- in-sample fit: number of parameters, R², adjusted R², AIC, BIC (lower is better)
- MAE and RMSE on the **validation** weeks, in pickups per zone-hour. Convert predictions back from logs with
  `np.expm1(prediction)`.

Pick the model with the lowest validation RMSE for the forecast-accuracy section.

## Forecast accuracy: MAE and RMSE on the held-out test weeks

- Refit the chosen model on train + validation and score it on the **test** weeks. Don't use the test weeks for
  fitting or for choosing the model.
- Compare it with the **last-week baseline**. If the regression can't beat it, it isn't adding much.
- Show the overall MAE / RMSE, a per-zone table for the busiest zones, and the share of zones where the model beats
  the baseline.
- Plot one busy zone's test week: actual vs model vs baseline.

Note: `exp(ŷ) − 1` predicts a median-like value, so it slightly under-predicts the mean.

## Frisch-Waugh-Lovell: how much does weather change demand, after the calendar pattern? 

Use city-wide hourly demand (sum over all 263 zones), training hours only.
- **Y** = log(total pickups in the hour)
- **X** = weather: temperature, precipitation, snowfall
- **Z** = calendar controls: hour of day, day of week, month dummies + constant (`patsy.dmatrix` works well)

Steps:
1. Regress Y on Z and keep the residuals: the demand the calendar doesn't explain.
2. Regress each X on Z and keep the residuals: weather that is unusual for that hour, weekday and month.
3. Regress the Y residuals on the X residuals. The slopes are the weather effects.

**Check:** the step-3 slopes must equal the weather coefficients in the full regression Y ~ X + Z (difference ≈ 0).
Take standard errors and p-values from the **full** regression with HAC errors, because the step-3 standard errors
use the wrong degrees of freedom. Report the effects as % change in pickups: `100 × (exp(coef) − 1)`.
Units: per 1 °C, per 1 mm of precipitation and per 1 cm of snowfall in the hour.

Add one partial-regression scatter per weather variable (X residual vs Y residual, with the fitted slope).

## Notes for the report

When reporting a result, state the model, the split and the standard-error type behind it.

- **Demand = completed pickups.** Unmet demand isn't observed, so busy hours (and the weather effect) are probably
  underestimated.
- **Actual weather is used,** but a real dispatch system would only have a forecast.
- **Models are fitted on the busiest zones only.** FWL uses the city-wide total.
- **Every zone shares one daily curve** in the additive ladder. The last-week baseline captures each zone's own
  shape, so it may beat the regression. Next things to try: zone × hour (and zone × weekend) interactions, or one
  model per zone group.
- **Temperature enters linearly.** Demand may rise in both cold and heat; `I(temperature_2m**2)` would test that.
- The cleaning decisions behind the data are in `docs/corrections.md`.